In [1]:
! pip install torch

In [2]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler

import torch

In [3]:
if torch.cuda.is_available():
    dev = "cuda:0"
elif torch.backends.mps.is_available():
    dev = "mps"
else:
    dev = "cpu"
device = torch.device(dev)
device

device(type='mps')

In [4]:
X, y = fetch_california_housing(return_X_y=True)

In [5]:
X

array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]], shape=(20640, 8))

The target variable is the median house value for California districts, expressed in hundreds of thousands of dollars ($100,000).

In [6]:
y

array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,))

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [8]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1).to(device)

In [10]:
train_ds = torch.utils.data.TensorDataset(X_train, y_train)
test_ds = torch.utils.data.TensorDataset(X_test, y_test)

In [11]:
train_ds, val_ds = torch.utils.data.random_split(train_ds, [0.9, 0.1])

In [12]:
mini_batch_size = 64

train_dl = torch.utils.data.DataLoader(train_ds, batch_size=mini_batch_size, shuffle=True, drop_last=False)
valid_dl = torch.utils.data.DataLoader(val_ds, batch_size=mini_batch_size * 2)

In [13]:
class FCN(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.fcn = torch.nn.Sequential(
            torch.nn.Linear(8, 20),
            torch.nn.ReLU(),
            torch.nn.Linear(20, 20),
            torch.nn.ReLU(),
            torch.nn.Linear(20, 10),
            torch.nn.ReLU(),
            torch.nn.Linear(10, 1)
        )

    def forward(self, X):
        X = self.fcn(X)
        return X

In [14]:
model = FCN().to(device)
optimizer = torch.optim.Adam(model.parameters())

In [15]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

821

In [16]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_model_state = None

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

    def load_best_model(self, model):
        model.load_state_dict(self.best_model_state)

In [17]:
early_stopping = EarlyStopping(patience=5, delta=0.01)

In [18]:
def fit(epochs, model, optimizer, train_dl, valid_dl=None):
    loss_func = torch.nn.MSELoss()

    # loop over epochs
    for epoch in range(epochs):
        model.train()

        # loop over mini-batches
        for X_mb, y_mb in train_dl:
            y_hat = model(X_mb)

            loss = loss_func(y_hat, y_mb)
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

        model.eval()
        with torch.no_grad():
            train_loss = sum(loss_func(model(X_mb), y_mb) for X_mb, y_mb in train_dl)
            valid_loss = sum(loss_func(model(X_mb), y_mb) for X_mb, y_mb in valid_dl)
        print('epoch {}, training loss {}'.format(epoch + 1, train_loss / len(train_dl)))
        print('epoch {}, validation loss {}'.format(epoch + 1, valid_loss / len(valid_dl)))

        early_stopping(valid_loss, model)
        if early_stopping.early_stop:
            print("Early stopping")
            break

    print('Finished training')

    return model

In [19]:
epochs = 50

fit(epochs, model, optimizer, train_dl, valid_dl)

epoch 1, training loss 0.7650843262672424
epoch 1, validation loss 0.6820886135101318
epoch 2, training loss 0.5072553753852844
epoch 2, validation loss 0.459007203578949
epoch 3, training loss 0.45622992515563965
epoch 3, validation loss 0.4100385308265686
epoch 4, training loss 0.4240696132183075
epoch 4, validation loss 0.39133113622665405
epoch 5, training loss 0.4073573648929596
epoch 5, validation loss 0.37970227003097534
epoch 6, training loss 0.394308477640152
epoch 6, validation loss 0.36864858865737915
epoch 7, training loss 0.383929580450058
epoch 7, validation loss 0.3636886179447174
epoch 8, training loss 0.37502625584602356
epoch 8, validation loss 0.3543100953102112
epoch 9, training loss 0.3756481111049652
epoch 9, validation loss 0.35500577092170715
epoch 10, training loss 0.3596172332763672
epoch 10, validation loss 0.3345012664794922
epoch 11, training loss 0.3546871542930603
epoch 11, validation loss 0.32956281304359436
epoch 12, training loss 0.3487277328968048
epo

FCN(
  (fcn): Sequential(
    (0): Linear(in_features=8, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=10, bias=True)
    (5): ReLU()
    (6): Linear(in_features=10, out_features=1, bias=True)
  )
)

In [20]:
early_stopping.load_best_model(model)

In [21]:
with torch.no_grad():
    yhat_train = model(train_ds[:][0]).cpu()

y_train = train_ds[:][1].cpu()

mean_absolute_error(y_train, yhat_train)

0.36672160029411316

In [22]:
with torch.no_grad():
    yhat_test = model(test_ds[:][0]).cpu()

y_test = test_ds[:][1].cpu()

mean_absolute_error(y_test, yhat_test)

0.3722047805786133